# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [1]:
!cat preprints.tsv

pub_date	title	venue	excerpt	abstract	citation	doi	url	url_slug	paper_url
2023-01-24	Improved Performance and Stability in Overflow Loss Systems via Exchange of Congestion Information	TechRxiv preprint	Trials the use of congestion information exchange as a form of admission control for stabilising the performance of overflow loss systems in which overflow traffic is penalised with longer average service times.	In systems and networks where overflow is penalized, careful planning is required to prevent a small increase in the offered load from suddenly triggering a system collapse. In this work, we consider an overflow loss system where the service time distribution of a request is dependent on the number of overflows. We propose a novel admission control policy for improving the stabilization and performance of such a system based on a new information exchange mechanism (IEM) for congestion information, in which only a small amount of congestion information is carried by each request a

## Import pandas

We are using the very handy pandas library for dataframes.

In [2]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [3]:
publications = pd.read_csv("preprints.tsv", sep="\t", header=0, encoding='utf-8')
publications


,pub_date,title,venue,excerpt,abstract,citation,doi,url,url_slug,paper_url
0,2023-01-24,Improved Performance and Stability in Overflow...,TechRxiv preprint,Trials the use of congestion information excha...,In systems and networks where overflow is pena...,"E. W. M. Wong and **Y. C. Chan**, “Improved pe...",10.36227/techrxiv.21908415.v1,NaN,wong2023_preprint,/files/wong2023_preprint.pdf
1,2023-08-28,Stability analysis of an epidemic model with t...,Research Square preprint,Proposes a two-strain epidemiological model wi...,The competition between pathogens is an essent...,"R. Niu, **Y.-C. Chan**, S. Liu, E. W. M. Wong,...",10.21203/rs.3.rs-3264948/v1,NaN,niu2023_preprint,/files/niu2023_preprint.pdf
2,2024-05-14,Building-Information-Modelling to Discrete-Eve...,SSRN preprint,Presents a framework for integrating building ...,Building Information Modelling (BIM) offers an...,"N. Moretti, **Y.-C. Chan**, M. Nakaoka, A. Muk...",10.2139/ssrn.4827727,NaN,moretti2024_preprint,/files/moretti2024_preprint.pdf


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [4]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [5]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: preprints"""
    
    md += """\npermalink: /preprints/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.abstract)) > 5:
        md += "\n**Abstract:** " + html_escape(item.abstract) + "\n"
    
    if len(str(item.doi)) > 5:
        md += "\n**DOI**: [" + item.doi + "](https://doi.org/" + item.doi + ")\n"
    elif len(str(item.url)) > 5:
        md += "\n**URL**: <" + item.url + ">\n" 
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\n**Recommended citation**:<br/>" + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_preprints/" + md_filename, 'w', encoding='utf-8') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [6]:
!ls ../_preprints/

2023-01-24-wong2023_preprint.md  2024-05-14-moretti2024_preprint.md
2023-08-28-niu2023_preprint.md


In [8]:
!cat ../_preprints/2024-05-14-moretti2024_preprint.md

---
title: "Building-Information-Modelling to Discrete-Event-Simulation Integration: Towards the Development of a Space-Process Digital Twin"
collection: preprints
permalink: /preprints/2024-05-14-moretti2024_preprint
excerpt: 'Presents a framework for integrating building information modelling (BIM) data and discrete-event simulation for a digital twin of a healthcare process.'
date: 2024-05-14
venue: 'SSRN preprint'
paperurl: '/files/moretti2024_preprint.pdf'
citation: 'N. Moretti, **Y.-C. Chan**, M. Nakaoka, A. Mukherjee, J. Merino, and A. K. Parlikad, “Building Information Modelling to Discrete Event Simulation Integration Towards the Development of a Space-Process Digital Twin”. *SSRN preprint*, May 2024.'
---
Presents a framework for integrating building information modelling (BIM) data and discrete-event simulation for a digital twin of a healthcare process.

**Abstract:** Building Information Modelling (BIM) offers an incredibly rich set of information that can be used across t